In [8]:
from workspace.sources.local_datasets.preprocessing.tokenization import NLTKTokenizer
from workspace.sources.local_datasets.preprocessing.cleaning import NoiseReduction, Lemmatization, Stemming
from workspace.sources.experiments.metrics import Loss
from workspace.sources.utils import class_name_to_str
from workspace.sources.experiments.experiment import Experiment
from workspace.sources.local_datasets.dataset import Dataset
from workspace.sources.local_datasets.preprocessing.pipelines import PreprocessingPipeline
from workspace.sources.local_datasets.preprocessing.utils import truncate
from sklearn.model_selection import ParameterGrid
from IPython.display import display, Markdown

from workspace.sources.local_datasets.preprocessing.encoders.encoders import GPT1_Encoder as Encoder
from workspace.sources.models.transformers.openai_gpt_models import GPT1 as Model

import mlflow

mlflow.set_tracking_uri('../../mlruns')

In [9]:
experiment_name = f'prefinal-{class_name_to_str(Model.__name__)}-v2'
print('Experiment:', experiment_name)

Experiment: prefinal-gpt1-v2


In [10]:
def conduct_gpt_model_experiments(dataset_name,
                                  dataset_path,
                                  dataset_hparams_grid,
                                  model_hparams_grid):
    total_runs = len(dataset_hparams_grid) * len(model_hparams_grid)
    for i, dataset_params in enumerate(dataset_hparams_grid):
        encoder = dataset_params['preprocessings_pipeline'][-1]
        for j, model_hparams in enumerate(model_hparams_grid):
            run_number = i * len(model_hparams_grid) + j + 1
            display(Markdown(f'### Run {run_number} of {total_runs}'))
            dataset = Dataset(dataset_name, dataset_path, **dataset_params)

            model = Model(gpt_encoder=encoder,
                          train_best_model_metric=Loss,
                          training_arguments=model_hparams)

            recovery_experiment = Experiment(
                name=experiment_name,
                dataset=dataset,
                model=model)
            recovery_experiment.run()


In [11]:

# max_tokens_params = [128, 512]
max_tokens_params = [128, 256]

pipelines = []

for max_tokens in max_tokens_params:
    pipelines.extend([PreprocessingPipeline(name='minimal',
                                            iterable=[Encoder(truncation_max_length=max_tokens)]),
                      PreprocessingPipeline(name='noise_reduction',
                                            iterable=[NoiseReduction(),
                                                      Encoder(truncation_max_length=max_tokens)]),
                      PreprocessingPipeline(name='noise_reduction_with_lemmatization',
                                            iterable=[NoiseReduction(), NLTKTokenizer(), Lemmatization(),
                                                      Encoder(is_split_into_words=True,
                                                              truncation_max_length=max_tokens)]),
                      PreprocessingPipeline(name='noise_reduction_with_stemming',
                                            iterable=[NoiseReduction(), NLTKTokenizer(), Stemming(),
                                                      Encoder(is_split_into_words=True,
                                                              truncation_max_length=max_tokens)])
                      ])
# optional - for testing purposes, if you want to run fast test on very small datasets
# truncate(pipelines, fraction=0.05)

dataset_hparams_grid = ParameterGrid({'preprocessings_pipeline': pipelines})

print('Dataset Hyperparameters Grid Size: ', len(dataset_hparams_grid))

model_hparams_grid = ParameterGrid({'epochs': [10],
                                    'batch_size': [16],
                                    'learning_rate': [1e-05, 5e-05],
                                    'lr_scheduler': ['linear'],
                                    'optimizer': ['adamw_torch'],
                                    'weight_decay': [0.0, 1e-3],
                                    'early_stopping_patience': [3],
                                    'early_stopping_threshold': [0.01]})

print('Model Hyperparameters Grid Size: ', len(model_hparams_grid))
print('Total Hyperparameter Combinations: ', len(dataset_hparams_grid) * len(model_hparams_grid))

Dataset Hyperparameters Grid Size:  4
Model Hyperparameters Grid Size:  1
Total Hyperparameter Combinations:  4


### ReCOVery Dataset

In [12]:
dataset_name = 'ReCOVery'
dataset_path = '../../sources/local_datasets/data/ReCOVery/recovery.csv'

conduct_gpt_model_experiments(dataset_name,
                              dataset_path,
                              dataset_hparams_grid,
                              model_hparams_grid)

### Run 1 of 4

2025/05/16 01:29:36 INFO mlflow.tracking.fluent: Experiment with name 'prefinal-gpt1-v2' does not exist. Creating a new experiment.
2025-05-16 01:29:36,856 - Experiment - INFO - MLflow experiment initialized with ID: 455212717504321152
2025-05-16 01:29:36,857 - Experiment - INFO - Model signature: model(mn=gpt1,ti=openai-gpt,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_torch,esp=3,est=0.01,pt=none,pti=none)
2025-05-16 01:29:36,858 - Experiment - INFO - Dataset signature: dataset(dn=recovery,tp=0.7,vp=0.15);pipeline([truncation(f=0.05,tt=top),gpt1__encoder(t=openai-gpt,t=true,p=max_length,tml=128,isiw=false,pt=none)])
2025-05-16 01:29:36,859 - Experiment - INFO - Run signature: model(mn=gpt1,ti=openai-gpt,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_torch,esp=3,est=0.01,pt=none,pti=none);dataset(dn=recovery,tp=0.7,vp=0.15);pipeline([truncation(f=0.05,tt=top),gpt1__encoder(t=openai-gpt,t=true,p=max_length,tml=128,isiw=false,pt=none)])
2025-05-16 01:29:

Map:   0%|          | 0/71 [00:00<?, ? examples/s]

Map:   0%|          | 0/71 [00:00<?, ? examples/s]

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

...

2025-05-16 01:40:58,669 - Experiment - INFO - Test metrics: {'test_loss': 0.5568803548812866, 'test_accuracy': 0.7333333333333333, 'test_precision': 0.7, 'test_recall': 0.875, 'test_f1_score': 0.7777777777777778, 'test_roc_auc': 0.7142857142857143, 'test_false_positives_rate': 0.42857142857142855, 'test_false_negatives_rate': 0.125, 'test_runtime': 0.9464, 'test_samples_per_second': 15.849, 'test_steps_per_second': 1.057, 'test_epoch': 4.0}
2025-05-16 01:40:58,689 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 01:40:58,691 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 01:40:58,742 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 01:40:58,743 - Experiment - INFO - Evaluating model using f1_score metric...
2025-05-16 01:40:58,744 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-10
2025-05-16 01:40:59,231 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.5549112558364868, 

2025-05-16 01:40:59,727 - Experiment - INFO - Test metrics: {'test_loss': 0.5691461563110352, 'test_accuracy': 0.6666666666666666, 'test_precision': 0.6153846153846154, 'test_recall': 1.0, 'test_f1_score': 0.7619047619047619, 'test_roc_auc': 0.8035714285714286, 'test_false_positives_rate': 0.7142857142857143, 'test_false_negatives_rate': 0.0, 'test_runtime': 0.4904, 'test_samples_per_second': 30.586, 'test_steps_per_second': 2.039, 'test_epoch': 2.0}
2025-05-16 01:40:59,744 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 01:40:59,746 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 01:40:59,790 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 01:40:59,791 - Experiment - INFO - Evaluating model using false_positives_rate metric...
2025-05-16 01:40:59,791 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-20
2025-05-16 01:41:00,275 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss'

2025-05-16 01:41:01,075 - Experiment - INFO - Test metrics: {'test_loss': 0.5568803548812866, 'test_accuracy': 0.7333333333333333, 'test_precision': 0.7, 'test_recall': 0.875, 'test_f1_score': 0.7777777777777778, 'test_roc_auc': 0.7142857142857143, 'test_false_positives_rate': 0.42857142857142855, 'test_false_negatives_rate': 0.125, 'test_runtime': 0.7945, 'test_samples_per_second': 18.879, 'test_steps_per_second': 1.259, 'test_epoch': 4.0}
2025-05-16 01:41:01,093 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 01:41:01,095 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 01:41:01,142 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 01:41:01,144 - Experiment - INFO - Evaluating model using roc_auc metric...
2025-05-16 01:41:01,145 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-20
2025-05-16 01:41:01,660 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.46003302931785583, 

2025-05-16 01:41:02,273 - Experiment - INFO - Test metrics: {'test_loss': 0.5568803548812866, 'test_accuracy': 0.7333333333333333, 'test_precision': 0.7, 'test_recall': 0.875, 'test_f1_score': 0.7777777777777778, 'test_roc_auc': 0.7142857142857143, 'test_false_positives_rate': 0.42857142857142855, 'test_false_negatives_rate': 0.125, 'test_runtime': 0.6074, 'test_samples_per_second': 24.695, 'test_steps_per_second': 1.646, 'test_epoch': 4.0}
2025-05-16 01:41:02,296 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 01:41:02,298 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 01:41:02,356 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 01:41:02,357 - Experiment - INFO - Evaluating model using loss metric...
2025-05-16 01:41:02,358 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-20
2025-05-16 01:41:02,848 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.46003302931785583, 'ev

2025-05-16 01:41:04,167 - Experiment - INFO - Test metrics: {'test_loss': 0.5568803548812866, 'test_accuracy': 0.7333333333333333, 'test_precision': 0.7, 'test_recall': 0.875, 'test_f1_score': 0.7777777777777778, 'test_roc_auc': 0.7142857142857143, 'test_false_positives_rate': 0.42857142857142855, 'test_false_negatives_rate': 0.125, 'test_runtime': 1.3105, 'test_samples_per_second': 11.446, 'test_steps_per_second': 0.763, 'test_epoch': 4.0}
2025-05-16 01:41:04,195 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 01:41:04,197 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 01:41:04,255 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 01:41:04,256 - Experiment - INFO - Finished model evaluations stage.


### EUvsDisinfo Dataset

In [ ]:
dataset_name = 'EUvsDisinfo'
dataset_path = '../../sources/local_datasets/data/EUvsDisinfo/euvsdisinfo.csv'

conduct_gpt_model_experiments(dataset_name,
                              dataset_path,
                              dataset_hparams_grid,
                              model_hparams_grid)

### ISOT Dataset

In [ ]:
dataset_name = 'ISOT'
dataset_path = '../../sources/local_datasets/data/ISOT/isot.csv'

conduct_gpt_model_experiments(dataset_name,
                              dataset_path,
                              dataset_hparams_grid,
                              model_hparams_grid)

### CZ Dataset

In [13]:
cz_pipelines = []

for max_tokens in max_tokens_params:
    cz_pipelines.extend([PreprocessingPipeline(name='minimal',
                                               iterable=[Encoder(truncation_max_length=max_tokens)]),
                         PreprocessingPipeline(name='noise_reduction',
                                               iterable=[NoiseReduction(),
                                                         Encoder(truncation_max_length=max_tokens)]),
                         PreprocessingPipeline(name='noise_reduction_with_lemmatization',
                                               iterable=[NoiseReduction(), NLTKTokenizer(language='czech'),
                                                         Lemmatization(language='czech'),
                                                         Encoder(is_split_into_words=True,
                                                                 truncation_max_length=max_tokens)]),
                         PreprocessingPipeline(name='noise_reduction_with_stemming',
                                               iterable=[NoiseReduction(), NLTKTokenizer(language='czech'),
                                                         Stemming(language='czech'),
                                                         Encoder(is_split_into_words=True,
                                                                 truncation_max_length=max_tokens)])
                         ])
# optional - for testing purposes, if you want to run fast test on very small datasets
# truncate(cz_pipelines, fraction=0.15)

cz_dataset_hparams_grid = ParameterGrid({'preprocessings_pipeline': cz_pipelines})
print('Dataset Hyperparameters Grid Size: ', len(cz_dataset_hparams_grid))
print('Model Hyperparameters Grid Size: ', len(model_hparams_grid))
print('Total Hyperparameter Combinations: ', len(cz_dataset_hparams_grid) * len(model_hparams_grid))

Dataset Hyperparameters Grid Size:  4
Model Hyperparameters Grid Size:  1
Total Hyperparameter Combinations:  4


In [14]:
dataset_name = 'cz_dataset'
dataset_path = '../../sources/local_datasets/data/cz_dataset/cz_dataset.csv'

conduct_gpt_model_experiments(dataset_name,
                              dataset_path,
                              cz_dataset_hparams_grid,
                              model_hparams_grid)

### Run 1 of 4

2025-05-16 01:41:04,957 - Experiment - INFO - MLflow experiment initialized with ID: 455212717504321152
2025-05-16 01:41:04,961 - Experiment - INFO - Model signature: model(mn=gpt1,ti=openai-gpt,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_torch,esp=3,est=0.01,pt=none,pti=none)
2025-05-16 01:41:04,963 - Experiment - INFO - Dataset signature: dataset(dn=cz_dataset,tp=0.7,vp=0.15);pipeline([truncation(f=0.15,tt=top),gpt1__encoder(t=openai-gpt,t=true,p=max_length,tml=128,isiw=false,pt=none)])
2025-05-16 01:41:04,964 - Experiment - INFO - Run signature: model(mn=gpt1,ti=openai-gpt,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_torch,esp=3,est=0.01,pt=none,pti=none);dataset(dn=cz_dataset,tp=0.7,vp=0.15);pipeline([truncation(f=0.15,tt=top),gpt1__encoder(t=openai-gpt,t=true,p=max_length,tml=128,isiw=false,pt=none)])
2025-05-16 01:41:06,012 - Experiment - INFO - Run ID: 417f52cf0974427b9c87e6dbe4103ad8
2025-05-16 01:41:06,014 - Experiment - INFO - Run name: mo

Map:   0%|          | 0/104 [00:00<?, ? examples/s]

Map:   0%|          | 0/104 [00:00<?, ? examples/s]

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

...

2025-05-16 01:48:32,160 - Experiment - INFO - Test metrics: {'test_loss': 0.7086583971977234, 'test_accuracy': 0.3181818181818182, 'test_precision': 0.3333333333333333, 'test_recall': 0.15384615384615385, 'test_f1_score': 0.21052631578947367, 'test_roc_auc': 0.41880341880341887, 'test_false_positives_rate': 0.4444444444444444, 'test_false_negatives_rate': 0.8461538461538461, 'test_runtime': 0.2571, 'test_samples_per_second': 85.568, 'test_steps_per_second': 7.779, 'test_epoch': 1.0}
2025-05-16 01:48:32,188 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 01:48:32,192 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 01:48:32,260 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 01:48:32,261 - Experiment - INFO - Evaluating model using f1_score metric...
2025-05-16 01:48:32,262 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-28
2025-05-16 01:48:32,768 - Experiment - INFO - Best entry according to validation 

2025-05-16 01:48:32,976 - Experiment - INFO - Test metrics: {'test_loss': 1.2725361585617065, 'test_accuracy': 0.6818181818181818, 'test_precision': 0.65, 'test_recall': 1.0, 'test_f1_score': 0.7878787878787878, 'test_roc_auc': 0.5042735042735043, 'test_false_positives_rate': 0.7777777777777778, 'test_false_negatives_rate': 0.0, 'test_runtime': 0.2034, 'test_samples_per_second': 108.178, 'test_steps_per_second': 9.834, 'test_epoch': 4.0}
2025-05-16 01:48:32,992 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 01:48:32,994 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 01:48:33,058 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 01:48:33,059 - Experiment - INFO - Evaluating model using false_positives_rate metric...
2025-05-16 01:48:33,060 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-7
2025-05-16 01:48:33,637 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.6363430619

2025-05-16 01:48:33,855 - Experiment - INFO - Test metrics: {'test_loss': 0.7086583971977234, 'test_accuracy': 0.3181818181818182, 'test_precision': 0.3333333333333333, 'test_recall': 0.15384615384615385, 'test_f1_score': 0.21052631578947367, 'test_roc_auc': 0.41880341880341887, 'test_false_positives_rate': 0.4444444444444444, 'test_false_negatives_rate': 0.8461538461538461, 'test_runtime': 0.213, 'test_samples_per_second': 103.308, 'test_steps_per_second': 9.392, 'test_epoch': 1.0}
2025-05-16 01:48:33,876 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 01:48:33,879 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 01:48:33,942 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 01:48:33,943 - Experiment - INFO - Evaluating model using roc_auc metric...
2025-05-16 01:48:33,944 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-7
2025-05-16 01:48:34,557 - Experiment - INFO - Best entry according to validation me

2025-05-16 01:48:34,772 - Experiment - INFO - Test metrics: {'test_loss': 0.7086583971977234, 'test_accuracy': 0.3181818181818182, 'test_precision': 0.3333333333333333, 'test_recall': 0.15384615384615385, 'test_f1_score': 0.21052631578947367, 'test_roc_auc': 0.41880341880341887, 'test_false_positives_rate': 0.4444444444444444, 'test_false_negatives_rate': 0.8461538461538461, 'test_runtime': 0.2075, 'test_samples_per_second': 106.026, 'test_steps_per_second': 9.639, 'test_epoch': 1.0}
2025-05-16 01:48:34,792 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 01:48:34,795 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 01:48:34,859 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 01:48:34,860 - Experiment - INFO - Evaluating model using loss metric...
2025-05-16 01:48:34,861 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-14
2025-05-16 01:48:35,469 - Experiment - INFO - Best entry according to validation met

2025-05-16 01:48:35,684 - Experiment - INFO - Test metrics: {'test_loss': 0.8452208042144775, 'test_accuracy': 0.5909090909090909, 'test_precision': 0.6, 'test_recall': 0.9230769230769231, 'test_f1_score': 0.7272727272727273, 'test_roc_auc': 0.5213675213675214, 'test_false_positives_rate': 0.8888888888888888, 'test_false_negatives_rate': 0.07692307692307693, 'test_runtime': 0.2086, 'test_samples_per_second': 105.459, 'test_steps_per_second': 9.587, 'test_epoch': 2.0}
2025-05-16 01:48:35,701 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 01:48:35,703 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 01:48:35,767 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 01:48:35,768 - Experiment - INFO - Finished model evaluations stage.
